In [1]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GATConv, to_hetero
from torch_geometric.loader import DataLoader
from halide_gnn_cost_model.data import PipelineDataset
from pathlib import Path

/opt/miniconda3/envs/pyg_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/miniconda3/envs/pyg_env/lib/python3.10/site-packages/torchtext/vocab/__init__.py:4: UserWarning: 
/!\ IMPORTANT WARNING ABOUT TORCHTEXT STATUS /!\ 
Torchtext is deprecated and the last released version will be 0.18 (this one). You can silence this warning by calling the following at the beginnign of your scripts: `import torchtext; torchtext.disable_torchtext_deprecation_warning()`
  warnings.warn(torchtext._TORCHTEXT_DEPRECATION_MSG)
/opt/miniconda3/envs/pyg_env/lib/python3.10/site-packages/torchtext/utils.py:4: UserWarning: 
/!\ IMPORTANT WARNING ABOUT TORCHTEXT STATUS /!\ 
Torchtext is deprecated and the last released version will be 0.18 (this one). You can silence this warning by calling the following at the beginnign

In [2]:
# Load dataset
dataset = PipelineDataset(Path("resources/pipelines"))
data = dataset[0]  # Get the first pipeline graph
print(data.metadata)

<bound method HeteroData.metadata of HeteroData(
  y=[5],
  function={ x=[6, 1] },
  ast_node={ x=[40, 1] },
  loop_level={ x=[25, 1] },
  (function, called_by, function)={ edge_index=[2, 7] },
  (ast_node, child_of, ast_node)={ edge_index=[2, 34] },
  (ast_node, is_expr_of, function)={ edge_index=[2, 6] },
  (loop_level, child_of, loop_level)={ edge_index=[2, 24] },
  (function, schedule_at, loop_level)={ edge_index=[2, 6] }
)>


In [3]:
data

HeteroData(
  y=[5],
  function={ x=[6, 1] },
  ast_node={ x=[40, 1] },
  loop_level={ x=[25, 1] },
  (function, called_by, function)={ edge_index=[2, 7] },
  (ast_node, child_of, ast_node)={ edge_index=[2, 34] },
  (ast_node, is_expr_of, function)={ edge_index=[2, 6] },
  (loop_level, child_of, loop_level)={ edge_index=[2, 24] },
  (function, schedule_at, loop_level)={ edge_index=[2, 6] }
)

In [8]:
class PipeGAT(torch.nn.Module):
    def __init__(self, hidden_channels, out_channels, num_layers=3):
        super(PipeGAT, self).__init__()
        self.convs = torch.nn.ModuleList()
        
        # --- FIX APPLIED HERE: add_self_loops=False ---
        self.convs.append(GATConv(-1, hidden_channels, heads=1, add_self_loops=False))
        
        for _ in range(num_layers - 2):
            self.convs.append(GATConv(-1, hidden_channels, heads=1, add_self_loops=False))
            
        self.convs.append(GATConv(-1, out_channels, heads=1, add_self_loops=False))

    def forward(self, x, edge_index):
        for conv in self.convs[:-1]:
            # You can ignore the PyG UserWarning about dropout/relu, but keeping them here:
            x = conv(x, edge_index)
            x = F.relu(x)
            x = F.dropout(x, p=0.5, training=self.training)
        x = self.convs[-1](x, edge_index)
        return x

In [9]:
gat = PipeGAT(hidden_channels=32, out_channels=32, num_layers=3)
gat = to_hetero(gat, data.metadata(), aggr="sum")

/opt/miniconda3/envs/pyg_env/lib/python3.10/site-packages/torch_geometric/nn/to_hetero_transformer.py:120: UserWarning: Found function 'dropout' with keyword argument 'training'. During FX tracing, this will likely be baked in as a constant value. Consider replacing this function by a module to properly encapsulate its training flag.
  return transformer.transform()
/opt/miniconda3/envs/pyg_env/lib/python3.10/site-packages/torch_geometric/nn/to_hetero_transformer.py:120: UserWarning: Found function 'dropout_1' with keyword argument 'training'. During FX tracing, this will likely be baked in as a constant value. Consider replacing this function by a module to properly encapsulate its training flag.
  return transformer.transform()


In [10]:
out = gat(data.x_dict, data.edge_index_dict)
out["function"][0]

tensor([-0.9154,  0.3284,  2.2838,  0.2167, -0.5053, -0.6815,  0.2267,  2.8170,
        -1.8230, -0.3373,  1.6877, -1.8272, -0.0982,  1.1819, -0.3025, -0.5917,
         1.4503, -2.7055, -1.0455, -1.6692, -1.4783, -1.3568,  0.3966, -1.3082,
         1.0880, -0.4736, -0.3987, -1.6460, -0.0692, -0.4534, -1.3880, -1.5926],
       grad_fn=<SelectBackward0>)

In [11]:
class PipelineModel(torch.nn.Module):
    def __init__(self, gnn, out_channels, num_runtime):
        super(PipelineModel, self).__init__()
        self.function_gnn = gnn
        self.pipeline_lin = torch.nn.Linear(out_channels, num_runtime)

    def forward(self, data, ptr=None):
        x_dict = data.x_dict
        edge_index_dict = data.edge_index_dict
        out = self.function_gnn(x_dict, edge_index_dict)
        # Get the feature of the pipeline node
        idx = 0 if ptr is None else ptr
        pipeline_feat = out["function"][idx]
        # Predict the runtime
        x = self.pipeline_lin(pipeline_feat)
        run_time = torch.exp(x)
        return run_time

In [12]:
model = PipelineModel(gat, 32, 5)
model

PipelineModel(
  (function_gnn): GraphModule(
    (convs): ModuleList(
      (0-2): 3 x ModuleDict(
        (function__called_by__function): GATConv(-1, 32, heads=1)
        (ast_node__child_of__ast_node): GATConv(-1, 32, heads=1)
        (ast_node__is_expr_of__function): GATConv(-1, 32, heads=1)
        (loop_level__child_of__loop_level): GATConv(-1, 32, heads=1)
        (function__schedule_at__loop_level): GATConv(-1, 32, heads=1)
      )
    )
  )
  (pipeline_lin): Linear(in_features=32, out_features=5, bias=True)
)

In [13]:
data_loader = DataLoader(dataset, batch_size=4, shuffle=True)
data_loader

In [14]:
# Train loop (example)
optimizer = torch.optim.Adam(model.parameters(), lr=0.002)
criterion = torch.nn.L1Loss()

for epoch in range(100):
    model.train()
    total_loss = 0
    for batch in data_loader:
        optimizer.zero_grad()
        pred = model(batch, batch["function"].ptr[:-1])
        # Assuming batch.y contains the true runtimes
        loss = criterion(pred.reshape(-1), batch.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}, Loss: {total_loss/len(data_loader)}")

Epoch 10, Loss: 317.1393783535411
Epoch 20, Loss: 289.4211076154068
Epoch 30, Loss: 287.3680195374922
Epoch 40, Loss: 289.89307611733085
Epoch 50, Loss: 289.6226221798908
Epoch 60, Loss: 286.9624002493417
Epoch 70, Loss: 296.11474094277787
Epoch 80, Loss: 297.02196484686357
Epoch 90, Loss: 295.89937386116964
Epoch 100, Loss: 295.89496579660255


In [15]:
torch.set_printoptions(precision=4)
print(model(data))

tensor([  2.0730,   7.2625,  34.7549, 159.2431, 919.2559],
       grad_fn=<ExpBackward0>)
